# Finnish GEC — Error Distribution Analysis

Run this notebook **before training any models** to understand:
- Error type frequencies in synthetic and Revita data
- Test set stratification requirements
- Data quality assessment

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

In [ ]:
def load_jsonl(path):
    """Load JSONL file."""
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load synthetic data sources
data_sources = {
    'confusion_sets': '../data/synthetic/confusion_sets/train.jsonl',
    'random_ops': '../data/synthetic/random_ops/train.jsonl',
    'back_translated': '../data/synthetic/back_translated/train.jsonl',
    'llm_generated': '../data/synthetic/llm_generated/train.jsonl',
    'revita': '../data/revita/train.jsonl'
}

datasets = {}
for name, path in data_sources.items():
    if Path(path).exists():
        datasets[name] = load_jsonl(path)
        print(f"✓ Loaded {name}: {len(datasets[name])} examples")
    else:
        print(f"⚠ {name} not found at {path}")

## 2. Basic Statistics

In [ ]:
def compute_stats(data):
    """Compute basic statistics for a dataset."""
    stats = {
        'num_examples': len(data),
        'avg_sentence_length': 0,
        'avg_edits_per_sentence': 0,
        'edit_types': Counter()
    }
    
    sentence_lengths = []
    edit_counts = []
    
    for example in data:
        sentence_lengths.append(len(example['corrupted'].split()))
        
        if 'edits' in example:
            edit_counts.append(len(example['edits']))
            for edit in example['edits']:
                stats['edit_types'][edit.get('type', 'unknown')] += 1
    
    stats['avg_sentence_length'] = sum(sentence_lengths) / len(sentence_lengths) if sentence_lengths else 0
    stats['avg_edits_per_sentence'] = sum(edit_counts) / len(edit_counts) if edit_counts else 0
    
    return stats

# Compute stats for all datasets
all_stats = {name: compute_stats(data) for name, data in datasets.items()}

# Display as DataFrame
stats_df = pd.DataFrame(all_stats).T
print(stats_df[['num_examples', 'avg_sentence_length', 'avg_edits_per_sentence']])

## 3. Error Type Distribution

In [ ]:
# Plot error type distributions
fig, axes = plt.subplots(len(datasets), 1, figsize=(12, 4 * len(datasets)))

if len(datasets) == 1:
    axes = [axes]

for ax, (name, stats) in zip(axes, all_stats.items()):
    error_types = stats['edit_types']
    if error_types:
        df = pd.DataFrame(error_types.most_common(10), columns=['Type', 'Count'])
        df.plot(kind='barh', x='Type', y='Count', ax=ax, legend=False)
        ax.set_title(f'{name} — Top 10 Error Types')
        ax.set_xlabel('Count')
    else:
        ax.text(0.5, 0.5, 'No edit metadata available', ha='center', va='center')
        ax.set_title(f'{name} — No Error Type Data')

plt.tight_layout()
plt.show()

## 4. Error Type Categorization

Categorize errors into linguistic categories for stratified evaluation.

In [ ]:
ERROR_CATEGORIES = {
    'morphological': ['confusion_set', 'case', 'tense', 'number'],
    'syntactic': ['word_order', 'syntax'],
    'lexical': ['word_choice', 'vocabulary'],
    'agreement': ['subject_verb', 'noun_modifier'],
    'punctuation': ['punctuation', 'capitalization'],
    'other': ['char_insert', 'char_delete', 'char_swap', 'unknown']
}

def categorize_error_type(error_type):
    """Map error type to linguistic category."""
    for category, types in ERROR_CATEGORIES.items():
        if any(t in error_type.lower() for t in types):
            return category
    return 'other'

# Categorize and plot
for name, stats in all_stats.items():
    categorized = Counter()
    for error_type, count in stats['edit_types'].items():
        category = categorize_error_type(error_type)
        categorized[category] += count
    
    print(f"\n{name}:")
    total = sum(categorized.values())
    for category, count in categorized.most_common():
        pct = 100 * count / total if total > 0 else 0
        print(f"  {category:20s}: {count:6d} ({pct:5.1f}%)")

## 5. Test Set Stratification Recommendations

Based on error distribution, generate stratified test split.

In [ ]:
# This is a placeholder for stratified split logic
print("TODO: Implement stratified test set split based on error type frequencies")
print("Ensure all error types are represented proportionally in test set")

## 6. Recommendations

Summary of findings and next steps.

In [ ]:
print("""\n=== RECOMMENDATIONS ===\n
1. Primary training data: Use confusion_sets as main synthetic source
2. Supplement with: LLM-generated errors for high-quality realistic errors
3. Reserve Revita entirely for evaluation (stratified by error type)
4. Track per-error-type performance in all experiments
5. Watch for generalization on low-frequency error types
""")